## Problem 1: A DataLoader batch

In [1]:
import torch
from torch.utils.data import TensorDataset, DataLoader

X = torch.randn(100, 8)
y = torch.randint(0, 3, (100,))

dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=16)

batch_X, batch_y = next(iter(loader))

print("batch X shape:", tuple(batch_X.shape))
print("batch y shape:", tuple(batch_y.shape))

batch X shape: (16, 8)
batch y shape: (16,)


## Problem 2: Train and evaluate

In [2]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch.nn as nn

X, y = load_digits(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=0,
    stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

torch.manual_seed(0)

model = nn.Sequential(
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 10)
)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for step in range(150):
    pred = model(X_train)
    loss = loss_fn(pred, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

with torch.no_grad():
    test_logits = model(X_test)
    test_preds = test_logits.argmax(dim=1)

accuracy = (test_preds == y_test).float().mean().item()

print("test accuracy above 0.90:", accuracy > 0.90)

test accuracy above 0.90: True


## Problem 3: Save and reload give identical predictions

In [3]:
torch.manual_seed(0)

model1 = nn.Sequential(
    nn.Linear(4, 3),
    nn.ReLU(),
    nn.Linear(3, 2)
)

sample = torch.randn(5, 4)

pred1 = model1(sample)

torch.save(model1.state_dict(), "model_weights.pth")

model2 = nn.Sequential(
    nn.Linear(4, 3),
    nn.ReLU(),
    nn.Linear(3, 2)
)

model2.load_state_dict(torch.load("model_weights.pth"))

pred2 = model2(sample)

print("identical:", torch.allclose(pred1, pred2))

identical: True


## Problem 4: Read the overfitting onset

In [4]:
val_losses = [0.90, 0.61, 0.48, 0.52, 0.70]

best_epoch = val_losses.index(min(val_losses))

print(f"best epoch: {best_epoch}")

best epoch: 2


## Problem 5: A mismatch is a RuntimeError

In [5]:
a = torch.randn(2, 3)
b = torch.randn(4, 2)

try:
    result = a @ b
except RuntimeError as error:
    print("error type:", type(error).__name__)

b_fixed = torch.randn(3, 2)
result = a @ b_fixed

print("after fix, shape:", tuple(result.shape))
print(
    "note: a device mismatch (model on GPU, batch on CPU) raises the same RuntimeError; "
    "the fix is to move both to the same device"
)

error type: RuntimeError
after fix, shape: (2, 2)
note: a device mismatch (model on GPU, batch on CPU) raises the same RuntimeError; the fix is to move both to the same device


## Problem 6: Reproducibility

In [6]:
def random_value(seed):
    torch.manual_seed(seed)
    return float(torch.rand(1))

first = random_value(42)
second = random_value(42)

print("runs match:", first == second)

runs match: True
